# 02 — scGPT embeddings and retrieval

Sanity-check the cached scGPT embeddings and exercise the FAISS indices
that power the `+retrieval` ablation level.

Prerequisites: `scripts/build_embeddings.py`,
`scripts/build_pathway_sets.py`, `scripts/build_retrieval_index.py`.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("project root:", ROOT)


## Cached embeddings

In [ ]:
import json
import numpy as np

meta = json.loads((ROOT / "data/embeddings/metadata.json").read_text())
print(json.dumps(meta, indent=2))

gene = np.load(ROOT / "data/embeddings/gene_embeddings.npz", allow_pickle=True)
cell = np.load(ROOT / "data/embeddings/cell_embeddings.npz", allow_pickle=True)
print("gene embeddings:", gene["embeddings"].shape)
print("cell embeddings:", cell["embeddings"].shape)

## Embedding sanity

Embeddings should be finite and roughly centred; wildly different norms
between genes would make the inner-product index meaningless.

In [ ]:
rng = np.random.default_rng(0)
sub = rng.choice(len(cell["embeddings"]), size=min(2000, len(cell["embeddings"])), replace=False)
emb = cell["embeddings"][sub]
norms = np.linalg.norm(emb, axis=1)
print(f"finite: {np.isfinite(emb).all()}")
print(f"mean={emb.mean():.4f}  std={emb.std():.4f}")
print(f"row-norm  median={np.median(norms):.3f}  p05={np.percentile(norms, 5):.3f}  p95={np.percentile(norms, 95):.3f}")

## Retrieval index queries

The perturbation index stores the summed scGPT gene embedding per perturbation
(L2-normalised, cosine similarity via inner product).

In [ ]:
import sys
sys.path.insert(0, str(ROOT / "src"))

from perturbgpt.agent import tools

index = tools.load_index("perturbation")
print("entries:", index.n_entries, "dim:", index.dim)

for pert in ["KLF1", "CEBPA"]:
    if pert not in index.ids:
        print(pert, "not in index")
        continue
    print()
    print(f"{pert} ->")
    for pid, score, _meta in index.query(pert, k=5, exclude_self=True):
        print(f"  {pid:<24s} {score:.4f}")

## Pathway ground truth

Retrieval is scored against MSigDB Hallmark co-membership. This shows how
many perturbations actually have a non-empty ground-truth neighbourhood.

In [ ]:
from perturbgpt.retrieval.metrics import build_pathway_co_membership

gt = build_pathway_co_membership(index.ids)
sizes = np.array([len(v) for v in gt.values()])
print(f"perturbations: {len(gt)}")
print(f"with non-empty GT: {(sizes > 0).sum()}")
non_empty = sizes[sizes > 0]
print("median neighbours (non-empty):", int(np.median(non_empty)) if len(non_empty) else 0)

results_path = ROOT / "data/indices/retrieval_results.json"
if results_path.exists():
    results = json.loads(results_path.read_text())
    print()
    print("K=5 recall (FAISS perturbation):",
          results["systems"]["faiss_perturbation"]["5"]["recall"])
    print("K=5 recall (random baseline):   ",
          results["systems"]["random_baseline"]["5"]["recall"])